In [1]:
import sys
from pathlib import Path
import os
import laspy
import numpy as np
import pandas as pd
import time 

In [9]:
from GlobalEvaluator2 import GlobalEvaluator, calcular_metricas_tesela
root = Path("../..").resolve()
sys.path.append(str(root))

from scripts.treeSegWatershed import clasify_tree_watershed


In [19]:
import pandas as pd
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt

def visualizar_arboles_watershed(df):
    """
    Visualiza una nube de puntos donde cada 'label' tiene un color distinto.
    """
    # 1. Extraer coordenadas y normalizar (importante para coordenadas UTM)
    points = df[['X', 'Y', 'Z']].values
    center = points.mean(axis=0)
    points_normalized = points - center

    # 2. Crear la nube de puntos de Open3D
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points_normalized)

    # 3. Asignar colores basados en la etiqueta (label)
    labels = df['label'].values
    unique_labels = np.unique(labels)
    
    # Generar una paleta de colores aleatorios
    # Usamos un mapa de colores de matplotlib para obtener colores distintivos
    cmap = plt.get_cmap("tab20") # 'tab20' tiene 20 colores bien diferenciados
    
    colors = np.zeros((len(labels), 3))
    
    for i, label in enumerate(unique_labels):
        # Si la etiqueta es -1 (usualmente suelo/ruido en Watershed), la ponemos en gris
        if label == -1:  # Consideramos -1 como fondo también
            colors[labels == label] = [0.2, 0.2, 0.2]
        else:
            # Asignar color según el índice del label único
            colors[labels == label] = cmap(i % 20)[:3]

    pcd.colors = o3d.utility.Vector3dVector(colors)

    # 4. Configurar el visualizador
    print(f"Visualizando {len(unique_labels)} etiquetas únicas.")
    vis = o3d.visualization.Visualizer()
    vis.create_window(window_name="Resultados Watershed - Santomera", width=1024, height=768)
    
    vis.add_geometry(pcd)
    
    # Mejorar la visualización (fondo oscuro y tamaño de punto)
    opt = vis.get_render_option()
    opt.background_color = np.asarray([0.05, 0.05, 0.05])
    opt.point_size = 2.0
    
    vis.run()
    vis.destroy_window()


In [3]:
def combine_df_las(arboles_df, tree_las_gt):
    # 2. Leer el LAS de Ground Truth
    points_las_gt = laspy.read(tree_las_gt)

    # 3. Crear un DataFrame temporal con las coordenadas del GT
    # Extraemos X, Y, Z del LAS para poder compararlos
    gt_df = pd.DataFrame({
        'X': np.array(points_las_gt.x),
        'Y': np.array(points_las_gt.y),
        'Z': np.array(points_las_gt.z)
    })


    arboles_df[['X', 'Y', 'Z']] = arboles_df[['X', 'Y', 'Z']]
    gt_df[['X', 'Y', 'Z']] = gt_df[['X', 'Y', 'Z']]

    
    arboles_df = arboles_df.drop_duplicates(subset=['X', 'Y', 'Z'])
    # 4. Filtrar usando un merge tipo 'inner'
    # Esto solo mantendrá las filas de arboles_df cuyas coordenadas existan en gt_df
    arboles_df = pd.merge(gt_df, arboles_df, on=['X', 'Y', 'Z'], how='left')

    # Los puntos que el watershed NO encontró aparecen como NaN. 
    # Al rellenarlos con 0, estás diciendo: "Mi modelo predijo que aquí NO hay árbol"
    arboles_df['label'] = arboles_df['label'].fillna(-1)

    return arboles_df

In [4]:
def generate_empty_result(ruta_trees):
    points_las_gt = laspy.read(ruta_trees)
    gt_df = pd.DataFrame({
        'X': np.array(points_las_gt.x),
        'Y': np.array(points_las_gt.y),
        'Z': np.array(points_las_gt.z)
    })
    gt_df['label'] = -1  # Asignar -1 a todos los puntos para indicar "no encontrado"
    return gt_df

In [ ]:
def process_and_test_watershed(ruta_trees,ruta_terrain,resolucion,altura_min_arbol,umbral_esfericidad, evaluator=None):    
 
    # 1. Aplicar watershed para segmentar árboles
    arboles_df = clasify_tree_watershed(ruta_trees, ruta_terrain, resolucion, altura_min_arbol, umbral_esfericidad)
    
    #print(f"visualizar_arboles_watershed para {ruta_trees}")
    if arboles_df.empty:
        arboles_df = generate_empty_result(ruta_trees)
        print("No se encontraron árboles con los parámetros dados para {0}.".format(ruta_trees))
        return
    # visualizar_arboles_watershed(arboles_df) 

    copia_arboles_df = arboles_df.copy()

    arboles_df = combine_df_las(arboles_df, ruta_trees)

    

    
    # # Crear métricas de evaluación
    res = calcular_metricas_tesela(arboles_df, ruta_trees)
    evaluator.acumular_tesela(res)

    return copia_arboles_df


In [ ]:
def process_folder_watershed(ruta_folder_trees, ruta_folder_terrain,resolucion,altura_min_arbol,umbral_esfericidad):
    evaluator = GlobalEvaluator()
    initial_time = time.time()

    
    for filename in os.listdir(ruta_folder_trees):
        if filename.endswith(".las"):
            ruta_trees = os.path.join(ruta_folder_trees, filename)
            ruta_terrain = os.path.join(ruta_folder_terrain, filename)
            process_and_test_watershed(ruta_trees,ruta_terrain,resolucion,
            altura_min_arbol,
            umbral_esfericidad, evaluator=evaluator)

    print(f"Numero de arboles predichos: {len(evaluator.all_pred_instances)}")
    end_time = time.time()

    return evaluator.reportar_micro_global(), (end_time - initial_time)/60

## VALIDACIÓN

In [19]:
grid_parameters = {
    'resolucion': np.linspace(0.1, 1.9, 10),  # Ejemplo: 0.5m, 1.0m, 1.5m, 2.0m
    'altura_min_arbol': np.linspace(0, 3, 6),  # Ejemplo: 0m, 1m, 2m, 3m, 4m, 5m
    'umbral_esfericidad': np.arange(0.00, 0.06, 0.01)  # Ejemplo: 0.00, 0.02, 0.04, 0.06, 0.08
}

print(f"Número de combinaciones a probar: {len(grid_parameters['resolucion']) * len(grid_parameters['altura_min_arbol']) * len(grid_parameters['umbral_esfericidad'])}")

Número de combinaciones a probar: 360


In [20]:
ruta_folder_trees = "../../data/Santomera/val/"
ruta_folder_terrain = "../../data/Santomera/tiles/terrain/"

mejor_report = {}, 0
mejor_configuracion = None

initial_time = time.time()

for resolucion in grid_parameters['resolucion']:
    for altura_min_arbol in grid_parameters['altura_min_arbol']:
        for umbral_esfericidad in grid_parameters['umbral_esfericidad']:
            print(f"Probando combinación: Resolución={resolucion}, Altura Mínima={altura_min_arbol}, Umbral Esfericidad={umbral_esfericidad}")
            report, tiempo = process_folder_watershed(ruta_folder_trees, ruta_folder_terrain, resolucion, altura_min_arbol, umbral_esfericidad)
            # Comparamos el ap50 de este reporte con el mejor encontrado hasta ahora
            if report[1] > mejor_report[1]:
                mejor_combinacion = (resolucion, altura_min_arbol, umbral_esfericidad)
                mejor_report = report
                print(f"¡Nueva mejor combinación encontrada! Resolución={resolucion}, Altura Mínima={altura_min_arbol}, Umbral Esfericidad={umbral_esfericidad} con AP50={report[1]:.4f}")
end_time = time.time()
print(f"Tiempo total de procesamiento: {(end_time - initial_time)/60:.2f} minutos")
print(f"Mejor combinación: Resolución={mejor_combinacion[0]}, Altura Mínima={mejor_combinacion[1]}, Umbral Esfericidad={mejor_combinacion[2]} con AP50={mejor_report[1]:.4f}")

Probando combinación: Resolución=0.1, Altura Mínima=0.0, Umbral Esfericidad=0.0
¡Nueva mejor combinación encontrada! Resolución=0.1, Altura Mínima=0.0, Umbral Esfericidad=0.0 con AP50=0.2002
Probando combinación: Resolución=0.1, Altura Mínima=0.0, Umbral Esfericidad=0.01
¡Nueva mejor combinación encontrada! Resolución=0.1, Altura Mínima=0.0, Umbral Esfericidad=0.01 con AP50=0.2203
Probando combinación: Resolución=0.1, Altura Mínima=0.0, Umbral Esfericidad=0.02
Probando combinación: Resolución=0.1, Altura Mínima=0.0, Umbral Esfericidad=0.03
Probando combinación: Resolución=0.1, Altura Mínima=0.0, Umbral Esfericidad=0.04
¡Nueva mejor combinación encontrada! Resolución=0.1, Altura Mínima=0.0, Umbral Esfericidad=0.04 con AP50=0.2370
Probando combinación: Resolución=0.1, Altura Mínima=0.0, Umbral Esfericidad=0.05
Probando combinación: Resolución=0.1, Altura Mínima=0.6, Umbral Esfericidad=0.0
Probando combinación: Resolución=0.1, Altura Mínima=0.6, Umbral Esfericidad=0.01
Probando combinació

## TEST

In [20]:
mejor_combinacion = (0.5, 1.2, 0.05)
ruta_folder_trees = "../../data/Santomera/tests/"
ruta_folder_terrain = "../../data/Santomera/tiles/terrain/"
report, tiempo = process_folder_watershed(ruta_folder_trees, ruta_folder_terrain, mejor_combinacion[0], mejor_combinacion[1], mejor_combinacion[2])
print(report)
print(f"Tiempo total de procesamiento: {tiempo:.2f} minutos")

Visualizando 10 etiquetas únicas.
Visualizando 9 etiquetas únicas.
Visualizando 8 etiquetas únicas.
Visualizando 10 etiquetas únicas.
Visualizando 7 etiquetas únicas.
Visualizando 7 etiquetas únicas.
Visualizando 4 etiquetas únicas.
Visualizando 7 etiquetas únicas.
Visualizando 7 etiquetas únicas.
Visualizando 5 etiquetas únicas.
Numero de arboles predichos: 64
   Categoría              Métrica     Valor Valor (%)
0  Semántica        IoU Semántico  0.721883    72.19%
1  Semántica   F1-Score Semántico  0.838481    83.85%
2  Instancia  Precision (Inst@50)  0.484375    48.44%
3  Instancia     Recall (Inst@50)  0.260504    26.05%
4  Instancia   F1-Score (Inst@50)  0.338798    33.88%
5  Instancia                mWCov  0.470268    47.03%
6  Instancia                 AP50  0.122258    12.23%
7  Instancia                 AP25  0.427757    42.78%
Tiempo total de procesamiento: 0.46 minutos
